### Autoencoder
Autoencoder learns the usual patterns in data like how normal transactions look.
When new transaction is given:
It tries to reconstruct the transaction from its previous knowledge on the transaction pattern.
If the transaction is normal, the reconstruction will be good.
If the transaction is strange / unusual, the reconstruction will be poor. Basically reconstruction error will be large. This will be flagged as anomaly.

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/transactions.csv")
df.head()

,transaction_id,customer_id,amount,hour,day_of_week,distance_from_home,merchant_category,is_fraud
0,TXN_00000000,861,321.657107,12,6,8.349037,restaurant,0
1,TXN_00000001,3773,1944.835714,18,2,4.499823,grocery,0
2,TXN_00000002,3093,14551.485133,11,2,18.237703,grocery,0
3,TXN_00000003,467,977.794274,6,1,2.069479,restaurant,0
4,TXN_00000004,4427,106.712706,20,1,13.467285,restaurant,0


### Feature Selection
The same numerical features used for Isolation Forest are used here to show comparison between the models.

In [5]:
features = [
    "amount",
    "hour",
    "day_of_week",
    "distance_from_home"
]
X = df[features]

In [6]:
# autoencoder requires feature scaling as it is depends on the feature magnitude
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
# training autoencode only on the normal transactions to 1st learns normal patterns
X_train = X_scaled[df["is_fraud"] == 0]

In [8]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(8, activation="relu")(input_layer)
decoded = Dense(input_dim, activation="linear")(encoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer="adam", loss="mse")

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 4)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 8)                   │              40 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │              36 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 76 (304.00 B)

 Trainable params: 76 (304.00 B)

 Non-trainable params: 0 (0.00 B)

to build a neural network ie autoencoder
Input: where data enters
Dense: layer which learns patterns
Model: connects everything into one model

shape[1] : How many features does each transaction have? The answer is 4 in my case.This tells all the input features the model needs automatically. 


### TensorFlow Not Installed
While I was importing TensorFlow, it resulted in a ModuleNotFoundError.
TensorFlow is not included by default in standard Python or Anaconda installations. So I had to install it separately.

In [5]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

ModuleNotFoundError: No module named 'tensorflow'

In [9]:
# to resolve error
import sys
print(sys.executable)

C:\ProgramData\anaconda3\python.exe


In [2]:
#to resolve error
# sanity check: confirming TensorFlow works before rerunning all the above cells, training autoencode

import tensorflow as tf
print(tf.__version__)

2.20.0


In [10]:
history = autoencoder.fit(
    X_train, X_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

Epoch 1/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0355 - val_loss: 0.5296
Epoch 2/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2987 - val_loss: 0.1478
Epoch 3/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0980 - val_loss: 0.0685
Epoch 4/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0502 - val_loss: 0.0364
Epoch 5/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0269 - val_loss: 0.0197
Epoch 6/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0142 - val_loss: 0.0099
Epoch 7/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0069 - val_loss: 0.0046
Epoch 8/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0034 - val_loss: 0.0024
Epoch 9/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0019 - val_loss: 0.0014
Epoch 10/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0011 - val_loss: 8.6630e-04


In [11]:
reconstructions = autoencoder.predict(X_scaled)
reconstruction_error = ((X_scaled - reconstructions) ** 2).mean(axis=1)

df["reconstruction_error"] = reconstruction_error
df["ae_predicted_fraud"] = (
    df["reconstruction_error"] > np.percentile(reconstruction_error, 99)
).astype(int)

df[["is_fraud", "ae_predicted_fraud"]].value_counts()

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step


is_fraud  ae_predicted_fraud
0         0                     97861
1         0                      1139
0         1                       882
1         1                       118
Name: count, dtype: int64

Correctly predicted normal transactions: 97,861--- actual users transactions were not disturbed

Fraud transactions predicted as normal: 1,139--- False -ves (fraud cases missed by the model)

Normal transactions predicted as fraudulent: 882--- False +ves (normal users incorrectly flagged)

Correctly detected fraud transactions: 118--- Fraud cases successfully identified by the autoencoder

### Observation
The autoencoder detected more fraud cases than Isolation Forest and One-Class SVM.

The number of false positives, false negatives, correctly predicted normal users, correctly predicted frauds are comparable to Isolation Forest and much better than one class svm.So, autoencoder is a slightly better in fraud detection.

But autoencoder mainly required: Neural network training,TensorFlow dependencies 